In [75]:
import os 
print(os.getcwd())

import glob

all_csv = glob.glob("*.csv")
print(all_csv)

C:\Users\Shawn\PROJECT\Stocksense\Reddit\sentiment
['cleaned_AAPL.csv', 'cleaned_AMZN.csv', 'cleaned_ARK.csv', 'cleaned_ATER.csv', 'cleaned_GOOGL.csv', 'cleaned_MSFT.csv', 'cleaned_RUI.csv', 'cleaned_TSLA.csv', 'cleaned_VXRT.csv']


In [76]:
ticketdf = pd.read_csv(file)
ticketdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29002 entries, 0 to 29001
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   text          29002 non-null  object
 1   date          29002 non-null  object
 2   likes         29002 non-null  int64 
 3   subreddit     29002 non-null  object
 4   cleaned_text  28988 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.1+ MB


# Flair

In [77]:
import pandas as pd
import flair
flair_sentiment = flair.models.TextClassifier.load('en-sentiment')  # Load model


def flair_senitment(ticketdf):
    for index, row in ticketdf.iterrows():
        s = flair.data.Sentence(row["cleaned_text"])  # Retrieve title of post
        flair_sentiment.predict(s) 
        sentiment = str(s.labels[0]).split()[0]
        score = str(s.labels[0]).split()[1][1:-1]

        ticketdf.at[index,'flair_sentiment'] = sentiment
        ticketdf.at[index,'flair_sentiment_score'] = score

    return ticketdf


        
 

2021-10-14 00:59:07,941 loading file C:\Users\Shawn\.flair\models\sentiment-en-mix-distillbert_4.pt


# Vader

In [78]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def vader_senitment(ticketdf):
    for index, row in ticketdf.iterrows():
        
        vs = sia.polarity_scores(test["cleaned_text"][0])
        
        score = vs["compound"]
        del vs['compound']
        sentiment = max(vs, key=vs.get)

        ticketdf.at[index,'vader_sentiment'] = sentiment
        ticketdf.at[index,'vader_score'] = score
    

    return ticketdf
    

# Finbert

In [79]:
import pandas as pd
from happytransformer import HappyTextClassification 
happy_tc = HappyTextClassification("BERT", "ProsusAI/finbert", num_labels=3)


def finbert_senitment(ticketdf):

    for index, row in ticketdf.iterrows():
        result = happy_tc.classify_text(row["cleaned_text"])
        sentiment = result.label
        score = result.score

        ticketdf.at[index,'finbert_sentiment'] = sentiment
        ticketdf.at[index,'finbert_score'] = score
    

    return ticketdf


10/14/2021 00:59:29 - INFO - happytransformer.happy_transformer -   Using model: cpu


In [83]:

for file in all_csv:
    file_df = pd.read_csv(file)
    print("======start======")
    file_df = flair_senitment(file_df) 
    print("flair done")
    file_df = vader_senitment(file_df)
    print("vader done")
    file_df = finbert_senitment(file_df)
    print("finbert done")
    filename = file.split(".")[0] + "_sentiment.csv"
    file_df.to_csv(filename,index=False)
    print(file,"done")


======start======


TypeError: expected string or bytes-like object